# CloudAura Fine-Tuning Pipeline
## LoRA/QLoRA SFT + DPO Preference Tuning

**Task:** JSON Extraction from natural language
**Base Model:** Qwen2.5-1.5B-Instruct
**Hardware:** Google Colab T4 GPU (free tier)

### Pipeline Steps
1. Install dependencies
2. Prepare datasets (SFT + DPO)
3. Evaluate base model (before)
4. SFT with QLoRA
5. DPO preference tuning
6. Evaluate all variants (after)
7. Export metrics for showcase site

### Free Tier Optimizations
- Reduced dataset sizes (2k SFT, 1k DPO)
- Single epoch training
- LoRA rank 8, shorter sequences (512)
- ~45 min total runtime on T4

In [ ]:
# Cell 1: Install dependencies
!pip install -q torch transformers peft trl datasets accelerate bitsandbytes sentencepiece protobuf

In [ ]:
# Cell 2: Verify GPU
import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"PyTorch: {torch.__version__}")

In [ ]:
# Cell 3: Configuration
import json
import os
import re
import time
from datetime import datetime, timezone

CONFIG = {
    "base_model": "Qwen/Qwen2.5-1.5B-Instruct",
    "task": "json_extraction",
    # SFT
    "sft_dataset": "HuggingFaceH4/ultrachat_200k",
    "sft_train_split": "train_sft",
    "sft_eval_split": "test_sft",
    "sft_max_train": 2000,
    "sft_max_eval": 200,
    "sft_epochs": 1,
    "sft_batch_size": 4,
    "sft_grad_accum": 4,
    "sft_lr": 2e-4,
    "sft_max_seq_len": 512,
    # DPO
    "dpo_dataset": "argilla/distilabel-intel-orca-dpo-pairs",
    "dpo_max_train": 1000,
    "dpo_epochs": 1,
    "dpo_batch_size": 2,
    "dpo_grad_accum": 4,
    "dpo_lr": 5e-5,
    "dpo_beta": 0.1,
    # LoRA
    "lora_r": 8,
    "lora_alpha": 16,
    "lora_dropout": 0.05,
    "target_modules": ["q_proj", "v_proj", "o_proj", "up_proj"],
}

JSON_EXTRACTION_SYSTEM = (
    "You are a precise JSON extraction assistant. Given a user's natural language "
    "input, extract the relevant information and return it as a valid JSON object. "
    "Only output JSON — no explanations, no markdown fences."
)

print("Config loaded.")
print(f"Base model: {CONFIG['base_model']}")
print(f"SFT samples: {CONFIG['sft_max_train']} (1 epoch)")
print(f"DPO samples: {CONFIG['dpo_max_train']} (1 epoch)")
print(f"LoRA rank: {CONFIG['lora_r']}, seq_len: {CONFIG['sft_max_seq_len']}")

In [ ]:
# Cell 4: Prepare SFT Dataset
from datasets import load_dataset

def format_chat_for_sft(example):
    messages = example.get("messages", [])
    if len(messages) < 2:
        return {"messages": []}
    user_msg = messages[0].get("content", "") if messages[0]["role"] == "user" else ""
    asst_msg = messages[1].get("content", "") if messages[1]["role"] == "assistant" else ""
    if not user_msg or not asst_msg:
        return {"messages": []}
    return {
        "messages": [
            {"role": "system", "content": JSON_EXTRACTION_SYSTEM},
            {"role": "user", "content": user_msg},
            {"role": "assistant", "content": asst_msg},
        ]
    }

print("Loading SFT dataset...")
sft_train = load_dataset(CONFIG["sft_dataset"], split=CONFIG["sft_train_split"])
sft_eval = load_dataset(CONFIG["sft_dataset"], split=CONFIG["sft_eval_split"])

sft_train = sft_train.shuffle(seed=42).select(range(min(CONFIG["sft_max_train"], len(sft_train))))
sft_eval = sft_eval.shuffle(seed=42).select(range(min(CONFIG["sft_max_eval"], len(sft_eval))))

sft_train = sft_train.map(format_chat_for_sft, remove_columns=sft_train.column_names)
sft_train = sft_train.filter(lambda x: len(x.get("messages", [])) > 0)
sft_eval = sft_eval.map(format_chat_for_sft, remove_columns=sft_eval.column_names)
sft_eval = sft_eval.filter(lambda x: len(x.get("messages", [])) > 0)

print(f"SFT Train: {len(sft_train)} | Eval: {len(sft_eval)}")

In [ ]:
# Cell 5: Prepare DPO Dataset
print("Loading DPO dataset...")
dpo_raw = load_dataset(CONFIG["dpo_dataset"], split="train")
dpo_raw = dpo_raw.filter(lambda x: x.get("status", "") != "tie")
dpo_raw = dpo_raw.shuffle(seed=42).select(range(min(CONFIG["dpo_max_train"], len(dpo_raw))))

def format_dpo(example):
    return {
        "prompt": [
            {"role": "system", "content": JSON_EXTRACTION_SYSTEM},
            {"role": "user", "content": example.get("input", "")},
        ],
        "chosen": [{"role": "assistant", "content": example.get("chosen", "")}],
        "rejected": [{"role": "assistant", "content": example.get("rejected", "")}],
    }

dpo_ds = dpo_raw.map(format_dpo, remove_columns=dpo_raw.column_names)
dpo_split = dpo_ds.train_test_split(test_size=0.05, seed=42)
dpo_train = dpo_split["train"]
dpo_eval = dpo_split["test"]

print(f"DPO Train: {len(dpo_train)} | Eval: {len(dpo_eval)}")

In [ ]:
# Cell 6: Evaluation prompts & scoring functions
EVAL_PROMPTS = [
    {
        "id": "eval_001",
        "input": "Cancel subscription for user ID 8842 effective immediately and send a confirmation email.",
        "expected": {"action": "cancel_subscription", "user_id": "8842", "effective": "immediately", "send_confirmation": True},
    },
    {
        "id": "eval_002",
        "input": "Transfer $500 from checking account to savings account for customer John Smith.",
        "expected": {"action": "transfer_funds", "amount": 500, "from_account": "checking", "to_account": "savings", "customer": "John Smith"},
    },
    {
        "id": "eval_003",
        "input": "Deploy version 2.4.1 of the payment-service to production in us-east-1 with a 10% canary rollout.",
        "expected": {"action": "deploy", "service": "payment-service", "version": "2.4.1", "environment": "production", "region": "us-east-1", "rollout_strategy": "canary", "rollout_percentage": 10},
    },
    {
        "id": "eval_004",
        "input": "Add a DNS A record for api.example.com pointing to 192.168.1.100 with TTL of 300 seconds.",
        "expected": {"action": "add_dns_record", "type": "A", "hostname": "api.example.com", "value": "192.168.1.100", "ttl": 300},
    },
    {
        "id": "eval_005",
        "input": "Create a support ticket: priority high, category billing, assigned to finance team. Customer reports double charge on order #7891.",
        "expected": {"action": "create_ticket", "priority": "high", "category": "billing", "assigned_to": "finance team", "description": "double charge on order #7891"},
    },
    {
        "id": "eval_006",
        "input": "Register a webhook at https://hooks.example.com/payments for payment.completed and payment.failed events with HMAC-SHA256 signing.",
        "expected": {"action": "register_webhook", "url": "https://hooks.example.com/payments", "events": ["payment.completed", "payment.failed"], "signing": "HMAC-SHA256"},
    },
    {
        "id": "eval_007",
        "input": "Scale the worker pool to 8 instances with 4 vCPUs and 16GB RAM each in the analytics namespace.",
        "expected": {"action": "scale", "resource": "worker pool", "instances": 8, "vcpus": 4, "memory_gb": 16, "namespace": "analytics"},
    },
    {
        "id": "eval_008",
        "input": "Revoke API key ak_live_abc123 for application 'mobile-app' and notify the developer at dev@company.io.",
        "expected": {"action": "revoke_api_key", "key_id": "ak_live_abc123", "application": "mobile-app", "notify": "dev@company.io"},
    },
]

def extract_json_from_text(text):
    text = text.strip()
    m = re.search(r'```(?:json)?\s*([\s\S]*?)```', text)
    if m: text = m.group(1).strip()
    m = re.search(r'\{[\s\S]*\}', text)
    if m: text = m.group(0)
    try: return json.loads(text)
    except: return None

def score_extraction(predicted, expected):
    if predicted is None:
        return {"valid_json": False, "key_precision": 0, "key_recall": 0, "key_f1": 0, "value_accuracy": 0}
    exp_keys, pred_keys = set(expected.keys()), set(predicted.keys())
    tp = exp_keys & pred_keys
    prec = len(tp) / len(pred_keys) if pred_keys else 0
    rec = len(tp) / len(exp_keys) if exp_keys else 0
    f1 = (2 * prec * rec / (prec + rec)) if (prec + rec) else 0
    val_matches = sum(
        1 for k in tp
        if (isinstance(expected[k], list) and isinstance(predicted.get(k), list) and set(map(str, expected[k])) == set(map(str, predicted.get(k, []))))
        or str(expected[k]).lower() == str(predicted.get(k, '')).lower()
    )
    return {"valid_json": True, "key_precision": round(prec,4), "key_recall": round(rec,4), "key_f1": round(f1,4), "value_accuracy": round(val_matches/len(exp_keys),4) if exp_keys else 0}

def evaluate_model(model, tokenizer, label):
    print(f"\nEvaluating: {label}")
    results = []
    total_lat, total_tok = 0, 0
    for p in EVAL_PROMPTS:
        msgs = [{"role":"system","content":JSON_EXTRACTION_SYSTEM},{"role":"user","content":p["input"]}]
        text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(text, return_tensors="pt").to(model.device)
        t0 = time.perf_counter()
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=512, do_sample=False, pad_token_id=tokenizer.pad_token_id)
        lat = time.perf_counter() - t0
        resp = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()
        parsed = extract_json_from_text(resp)
        sc = score_extraction(parsed, p["expected"])
        tok = len(tokenizer.encode(resp))
        total_lat += lat; total_tok += tok
        results.append({"id":p["id"],"scores":sc,"latency":round(lat,3),"tokens":tok,"raw_output":resp,"parsed":parsed})
    n = len(results)
    summary = {
        "model": label,
        "valid_json_rate": round(sum(1 for r in results if r["scores"]["valid_json"])/n, 4),
        "avg_key_f1": round(sum(r["scores"]["key_f1"] for r in results)/n, 4),
        "avg_value_accuracy": round(sum(r["scores"]["value_accuracy"] for r in results)/n, 4),
        "avg_latency_s": round(total_lat/n, 3),
        "tokens_per_second": round(total_tok/total_lat, 1) if total_lat else 0,
    }
    print(f"  JSON valid: {summary['valid_json_rate']*100:.1f}% | Key F1: {summary['avg_key_f1']:.3f} | Value Acc: {summary['avg_value_accuracy']:.3f} | Latency: {summary['avg_latency_s']:.2f}s")
    return {"summary": summary, "results": results}

print(f"Evaluation setup: {len(EVAL_PROMPTS)} prompts ready.")

In [ ]:
# Cell 7: Load base model with QLoRA quantization
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

print(f"Loading {CONFIG['base_model']}...")
base_model = AutoModelForCausalLM.from_pretrained(
    CONFIG["base_model"],
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
tokenizer = AutoTokenizer.from_pretrained(CONFIG["base_model"], trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    base_model.config.pad_token_id = tokenizer.pad_token_id

print(f"Model loaded. Parameters: {sum(p.numel() for p in base_model.parameters()):,}")

In [ ]:
# Cell 8: Evaluate BASE model (before fine-tuning)
base_model.eval()
base_eval = evaluate_model(base_model, tokenizer, "Base Model (Qwen2.5-1.5B-Instruct)")

In [ ]:
# Cell 9: SFT Training with QLoRA
from trl.trainer.sft_trainer import SFTTrainer
from trl.trainer.sft_config import SFTConfig

sft_model = prepare_model_for_kbit_training(base_model)

lora_config = LoraConfig(
    r=CONFIG["lora_r"],
    lora_alpha=CONFIG["lora_alpha"],
    lora_dropout=CONFIG["lora_dropout"],
    target_modules=CONFIG["target_modules"],
    task_type="CAUSAL_LM",
    bias="none",
)
sft_model = get_peft_model(sft_model, lora_config)

trainable, total = sft_model.get_nb_trainable_parameters()
print(f"Trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

sft_args = SFTConfig(
    output_dir="./outputs/sft",
    num_train_epochs=CONFIG["sft_epochs"],
    per_device_train_batch_size=CONFIG["sft_batch_size"],
    per_device_eval_batch_size=CONFIG["sft_batch_size"],
    gradient_accumulation_steps=CONFIG["sft_grad_accum"],
    learning_rate=CONFIG["sft_lr"],
    weight_decay=0.01,
    warmup_steps=10,
    lr_scheduler_type="cosine",
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=2,
    bf16=True,
    gradient_checkpointing=True,
    optim="paged_adamw_8bit",
    report_to="none",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
)

sft_trainer = SFTTrainer(
    model=sft_model,
    args=sft_args,
    train_dataset=sft_train,
    eval_dataset=sft_eval,
    processing_class=tokenizer,
)

print("Starting SFT training...")
sft_result = sft_trainer.train()
print(f"SFT complete. Train loss: {sft_result.metrics.get('train_loss', 0):.4f}")

sft_trainer.save_model("./outputs/sft/final_adapter")
tokenizer.save_pretrained("./outputs/sft/final_adapter")
sft_metrics = {**sft_result.metrics, **sft_trainer.evaluate()}
print(f"Eval loss: {sft_metrics.get('eval_loss', 0):.4f}")

In [ ]:
# Cell 10: Evaluate SFT model
sft_model.eval()
sft_eval_results = evaluate_model(sft_model, tokenizer, "SFT (QLoRA)")

In [ ]:
# Cell 11: DPO Preference Tuning
from peft import PeftModel
from trl import DPOConfig, DPOTrainer

# Merge SFT adapter and prepare fresh LoRA for DPO
del sft_model
torch.cuda.empty_cache()

dpo_base = AutoModelForCausalLM.from_pretrained(
    CONFIG["base_model"],
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
dpo_base = PeftModel.from_pretrained(dpo_base, "./outputs/sft/final_adapter", is_trainable=False)
dpo_base = dpo_base.merge_and_unload()

dpo_lora_config = LoraConfig(
    r=CONFIG["lora_r"],
    lora_alpha=CONFIG["lora_alpha"],
    lora_dropout=CONFIG["lora_dropout"],
    target_modules=CONFIG["target_modules"],
    task_type="CAUSAL_LM",
    bias="none",
)

dpo_args = DPOConfig(
    output_dir="./outputs/dpo",
    num_train_epochs=CONFIG["dpo_epochs"],
    per_device_train_batch_size=CONFIG["dpo_batch_size"],
    per_device_eval_batch_size=CONFIG["dpo_batch_size"],
    gradient_accumulation_steps=CONFIG["dpo_grad_accum"],
    learning_rate=CONFIG["dpo_lr"],
    weight_decay=0.01,
    warmup_steps=5,
    lr_scheduler_type="cosine",
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=2,
    bf16=True,
    gradient_checkpointing=True,
    beta=CONFIG["dpo_beta"],
    loss_type="sigmoid",
    max_length=512,
    max_prompt_length=256,
    report_to="none",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
)

dpo_trainer = DPOTrainer(
    model=dpo_base,
    args=dpo_args,
    train_dataset=dpo_train,
    eval_dataset=dpo_eval,
    processing_class=tokenizer,
    peft_config=dpo_lora_config,
)

print("Starting DPO training...")
dpo_result = dpo_trainer.train()
print(f"DPO complete. Train loss: {dpo_result.metrics.get('train_loss', 0):.4f}")

dpo_trainer.save_model("./outputs/dpo/final_adapter")
tokenizer.save_pretrained("./outputs/dpo/final_adapter")
dpo_metrics = {**dpo_result.metrics, **dpo_trainer.evaluate()}
print(f"Eval loss: {dpo_metrics.get('eval_loss', 0):.4f}")

In [ ]:
# Cell 12: Evaluate DPO model
dpo_model = dpo_trainer.model
dpo_model.eval()
dpo_eval_results = evaluate_model(dpo_model, tokenizer, "SFT + DPO")

In [ ]:
# Cell 13: Generate final metrics report
report = {
    "timestamp": datetime.now(timezone.utc).isoformat(),
    "config": CONFIG,
    "hardware": {
        "gpu": torch.cuda.get_device_name(0),
        "vram_gb": round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1),
    },
    "training": {
        "sft": {
            "train_loss": sft_metrics.get("train_loss", 0),
            "eval_loss": sft_metrics.get("eval_loss", 0),
            "train_samples": len(sft_train),
            "epochs": CONFIG["sft_epochs"],
            "lora_rank": CONFIG["lora_r"],
            "trainable_params_pct": round(100 * trainable / total, 2),
        },
        "dpo": {
            "train_loss": dpo_metrics.get("train_loss", 0),
            "eval_loss": dpo_metrics.get("eval_loss", 0),
            "train_samples": len(dpo_train),
            "epochs": CONFIG["dpo_epochs"],
            "beta": CONFIG["dpo_beta"],
        },
    },
    "evaluation": {
        "base": base_eval["summary"],
        "sft": sft_eval_results["summary"],
        "dpo": dpo_eval_results["summary"],
    },
    "detailed_results": {
        "base": base_eval["results"],
        "sft": sft_eval_results["results"],
        "dpo": dpo_eval_results["results"],
    },
}

os.makedirs("./results", exist_ok=True)
with open("./results/eval_metrics.json", "w") as f:
    json.dump(report, f, indent=2, default=str)

print("\n" + "="*70)
print("FINAL COMPARISON")
print("="*70)
print(f"{'Model':<30} {'JSON%':>8} {'Key F1':>8} {'Val Acc':>8} {'Latency':>8}")
print("-"*70)
for key in ["base", "sft", "dpo"]:
    s = report["evaluation"][key]
    print(f"{s['model']:<30} {s['valid_json_rate']*100:>7.1f}% {s['avg_key_f1']:>8.3f} {s['avg_value_accuracy']:>8.3f} {s['avg_latency_s']:>7.2f}s")
print("="*70)
print("\nMetrics saved to ./results/eval_metrics.json")
print("Download this file and place it in your cloudaura-finetune/results/ directory.")